**Drs List on Big Query**

In [10]:
from google.cloud import bigquery
from google.oauth2 import service_account

# =========================================
# GOOGLE BIGQUERY AUTHENTICATION
# =========================================

SERVICE_ACCOUNT_FILE = "../Keys/BigQueryKey.json"

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE
)

# =========================================
# CONFIGURATION
# =========================================

PROJECT_ID = "depihealthnux"
DATASET_ID = "Depihealthnux_Main"
TABLE_ID = "Drs_List"

TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

# =========================================
# INITIALIZE BIGQUERY CLIENT
# =========================================

client = bigquery.Client(
    credentials=credentials,
    project=PROJECT_ID
)

# =========================================
# TABLE SCHEMA
# =========================================

schema = [
    # Primary Key
    bigquery.SchemaField(
        "Dr_Code",
        "STRING",
        mode="REQUIRED"
    ),

    # Basic Information
    bigquery.SchemaField("Speciality", "STRING"),
    bigquery.SchemaField("Dr_Name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("Email", "STRING"),

    # Financial Information
    bigquery.SchemaField("Visit_Fee", "INTEGER"),

    # Status
    bigquery.SchemaField("Active_Dr", "STRING"),

    # Dates
    bigquery.SchemaField("Joining_Date", "DATE")
]

# =========================================
# CREATE TABLE OBJECT
# =========================================

table = bigquery.Table(
    TABLE_REF,
    schema=schema
)

# =========================================
# CLUSTERING
# =========================================

# Clustering by speciality and dr_code 
table.clustering_fields = [
    "speciality",
    "dr_code"
]

# =========================================
# CREATE TABLE
# =========================================

table = client.create_table(
    table,
    exists_ok=True
)

print("=================================")
print("Table created successfully")
print(TABLE_REF)
print("=================================")

Table created successfully
depihealthnux.Depihealthnux_Main.Drs_List


**ETL & Fill Drs List by Data**

In [12]:
import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account

# =========================================
# GOOGLE BIGQUERY AUTHENTICATION
# =========================================

SERVICE_ACCOUNT_FILE = "../Keys/BigQueryKey.json"

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE
)

PROJECT_ID = "depihealthnux"
DATASET_ID = "Depihealthnux_Main"
TABLE_ID = "Drs_List"

TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

client = bigquery.Client(
    credentials=credentials,
    project=PROJECT_ID
)

# =========================================
# READ EXCEL FILE
# =========================================

df = pd.read_excel("../Resources/Drs_List.xlsx")

# =========================================
# GET LAST DR CODE FROM BIGQUERY
# =========================================

query = f"""
SELECT MAX(CAST(REGEXP_EXTRACT(Dr_Code, r'\\d+') AS INT64)) AS max_id
FROM `{TABLE_REF}`
"""

result = client.query(query).to_dataframe()

last_id = result.iloc[0]["max_id"]

if pd.isna(last_id):
    last_id = 0

# =========================================
# GENERATE NEW DR CODES
# =========================================

df.insert(
    0,
    "Dr_Code",
    [f"Dr{i:03d}" for i in range(last_id + 1, last_id + len(df) + 1)]
)

# =========================================
# CLEAN DATA TYPES
# =========================================

df["Joining_Date"] = pd.to_datetime(
    df["Joining_Date"]
).dt.date

df["Visit_Fee"] = pd.to_numeric(
    df["Visit_Fee"],
    errors="coerce"
)

# =========================================
# REORDER COLUMNS TO MATCH TABLE
# =========================================

df = df[
    [
        "Dr_Code",
        "Speciality",
        "Dr_Name",
        "Email",
        "Visit_Fee",
        "Active_Dr",
        "Joining_Date"
    ]
]

# =========================================
# LOAD TO BIGQUERY
# =========================================

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND"
)

job = client.load_table_from_dataframe(
    df,
    TABLE_REF,
    job_config=job_config
)

job.result()

print("=================================")
print(f"{len(df)} rows inserted successfully")
print(TABLE_REF)
print("=================================")

20 rows inserted successfully
depihealthnux.Depihealthnux_Main.Drs_List


**Create Postgres table**

In [1]:
import sys
import psycopg2

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# =========================================
# CONNECT TO POSTGRES
# =========================================

conn = psycopg2.connect(POSTGRES_URL)
cursor = conn.cursor()

# =========================================
# CREATE SEQUENCE
# =========================================

cursor.execute("""

CREATE SEQUENCE IF NOT EXISTS dr_no_seq;

""")

# =========================================
# CREATE TABLE
# =========================================

create_table_query = """
CREATE TABLE IF NOT EXISTS Dr_List (

    -- Primary Key
    Dr_Code TEXT PRIMARY KEY
    DEFAULT (
        'Dr' ||
        LPAD(
            nextval('dr_no_seq')::TEXT,
            3,
            '0'
        )
    ),

    -- Basic Information
    Speciality TEXT,
    Dr_Name TEXT NOT NULL,
    Email TEXT,

    -- Financial Information
    Visit_Fee INTEGER,

    -- Status
    Active_Dr TEXT,

    -- Dates
    Joining_Date DATE

);
"""

cursor.execute(create_table_query)

# =========================================
# INDEXES
# =========================================

cursor.execute("""

CREATE INDEX IF NOT EXISTS idx_dr_speciality
ON Dr_List(Speciality);

""")

cursor.execute("""

CREATE INDEX IF NOT EXISTS idx_dr_active
ON Dr_List(Active_Dr);

""")

conn.commit()

print("=================================")
print("PostgreSQL table created successfully")
print("Table: Dr_List")
print("=================================")

cursor.close()
conn.close()

PostgreSQL table created successfully
Table: Dr_List


**Sync Big Query to Postgres**

In [2]:
import sys
import pandas as pd
import psycopg2

from psycopg2.extras import execute_values

from google.cloud import bigquery
from google.oauth2 import service_account

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# =========================================
# BIGQUERY AUTH
# =========================================

credentials = service_account.Credentials.from_service_account_file(
    "../Keys/BigQueryKey.json"
)

client = bigquery.Client(
    credentials=credentials,
    project="depihealthnux"
)

# =========================================
# READ BIGQUERY
# =========================================

query = """

SELECT

    Dr_Code,
    Speciality,
    Dr_Name,
    Email,
    Visit_Fee,
    Active_Dr,
    Joining_Date

FROM `depihealthnux.Depihealthnux_Main.Drs_List`

ORDER BY Dr_Code

"""

df = client.query(query).to_dataframe()

print(df.head(3))
print(f"Rows Retrieved: {len(df)}")

# =========================================
# CONNECT POSTGRES
# =========================================

conn = psycopg2.connect(POSTGRES_URL)
cursor = conn.cursor()

cursor.execute("""
DELETE FROM Dr_List;
""")

if not df.empty:

    execute_values(
        cursor,
        """
        INSERT INTO Dr_List (

            Dr_Code,
            Speciality,
            Dr_Name,
            Email,
            Visit_Fee,
            Active_Dr,
            Joining_Date

        )
        VALUES %s
        """,
        df.values.tolist(),
        page_size=1000
    )

# =========================================
# RESET SEQUENCE
# =========================================

cursor.execute("""

SELECT setval(
    'dr_no_seq',
    COALESCE(
        (
            SELECT MAX(
                CAST(
                    REPLACE(Dr_Code,'Dr','')
                    AS INTEGER
                )
            )
            FROM Dr_List
        ),
        1
    ),
    true
);

""")

conn.commit()

print(f"Inserted {len(df)} rows")

cursor.execute("""

SELECT *
FROM Dr_List
ORDER BY Dr_Code
LIMIT 3

""")

print("\nFirst 3 Rows From PostgreSQL")

for row in cursor.fetchall():
    print(row)

cursor.close()
conn.close()

  Dr_Code        Speciality    Dr_Name              Email  Visit_Fee  \
0   Dr001             Chest  Dr. Wafaa  drwafaa@gmail.com        450   
1   Dr002  Gastroenterology  Dr. Angie    angie@gmail.com        700   
2   Dr003      Rheumatology    Dr. Aml   amal@hotmail.com       1000   

  Active_Dr Joining_Date  
0    Active   2025-05-13  
1    Active   2025-05-13  
2    Active   2025-06-12  
Rows Retrieved: 20
Inserted 20 rows

First 3 Rows From PostgreSQL
('Dr001', 'Chest', 'Dr. Wafaa', 'drwafaa@gmail.com', 450, 'Active', datetime.date(2025, 5, 13))
('Dr002', 'Gastroenterology', 'Dr. Angie', 'angie@gmail.com', 700, 'Active', datetime.date(2025, 5, 13))
('Dr003', 'Rheumatology', 'Dr. Aml', 'amal@hotmail.com', 1000, 'Active', datetime.date(2025, 6, 12))


**Sync Postgres to BigQuery**

In [3]:
import sys
import pandas as pd
import psycopg2

from google.cloud import bigquery
from google.oauth2 import service_account

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# =========================================
# BIGQUERY AUTH
# =========================================

credentials = service_account.Credentials.from_service_account_file(
    "../Keys/BigQueryKey.json"
)

client = bigquery.Client(
    credentials=credentials,
    project="depihealthnux"
)

# =========================================
# READ POSTGRES
# =========================================

conn = psycopg2.connect(POSTGRES_URL)

query = """

SELECT

    Dr_Code,
    Speciality,
    Dr_Name,
    Email,
    Visit_Fee,
    Active_Dr,
    Joining_Date

FROM Dr_List

ORDER BY Dr_Code

"""

df = pd.read_sql(query, conn)

print(df.head(3))
print(f"Rows Retrieved: {len(df)}")

conn.close()

# =========================================
# LOAD TO BIGQUERY
# =========================================

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

job = client.load_table_from_dataframe(
    df,
    "depihealthnux.Depihealthnux_Main.Drs_List",
    job_config=job_config
)

job.result()

print(f"Uploaded {len(df)} rows")

# =========================================
# VERIFY
# =========================================

verify_df = client.query("""

SELECT *
FROM `depihealthnux.Depihealthnux_Main.Drs_List`
LIMIT 3

""").to_dataframe()

print("\nFirst 3 Rows From BigQuery")

print(verify_df)

C:\Users\Sedeek Elmasry\AppData\Local\Temp\ipykernel_14004\2004580147.py:48: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


  dr_code        speciality    dr_name              email  visit_fee  \
0   Dr001             Chest  Dr. Wafaa  drwafaa@gmail.com        450   
1   Dr002  Gastroenterology  Dr. Angie    angie@gmail.com        700   
2   Dr003      Rheumatology    Dr. Aml   amal@hotmail.com       1000   

  active_dr joining_date  
0    Active   2025-05-13  
1    Active   2025-05-13  
2    Active   2025-06-12  
Rows Retrieved: 20
Uploaded 20 rows

First 3 Rows From BigQuery
  dr_code       speciality     dr_name                  email  visit_fee  \
0   Dr020   Anesthesiology  Dr. Sherif  sherif.anes@gmail.com       1100   
1   Dr010  General Surgery  Dr. Khaled  khaled.surg@gmail.com       1000   
2   Dr011       Pediatrics   Dr. Laila   laila.peds@yahoo.com        500   

  active_dr joining_date  
0    Active   2025-02-02  
1    Active   2025-09-05  
2    Active   2025-08-22  
